# BÀI TOÁN TÌM KIẾM

## Bài toán mê cung

In [1]:
import heapq  

In [2]:
def heuristic(a, b):
    """
    Tính khoảng cách Manhattan giữa 2 điểm a và b
    a, b là tuple dạng (x, y)
    """
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


In [3]:
def Astar_maze(maze, start, goal):
    """
    maze  : ma trận 0 (đi được) và 1 (tường)
    start : vị trí bắt đầu (x, y)
    goal  : vị trí đích (x, y)
    
    Trả về:
    parent: mảng truy vết đường đi
    """
    
    rows = len(maze)
    cols = len(maze[0])
    
    OPEN = []                              # hàng đợi ưu tiên
    heapq.heappush(OPEN, (0, start))        # đưa start vào OPEN
    
    g = {start: 0}                          # g(p): chi phí từ start → p
    parent = {start: None}                  # lưu đỉnh trước đó
    CLOSED = set()                          # tập đỉnh đã duyệt
    
    while OPEN:
        _, current = heapq.heappop(OPEN)    # lấy đỉnh có f nhỏ nhất
        
        if current in CLOSED:               # nếu đã duyệt thì bỏ qua
            continue
        
        if current == goal:                 # nếu tới goal thì dừng
            break
        
        CLOSED.add(current)
        
        x, y = current
        # duyệt 4 hướng: lên, xuống, trái, phải
        for dx, dy in [(-1,0),(1,0),(0,-1),(0,1)]:
            nx, ny = x + dx, y + dy
            
            # kiểm tra nằm trong biên
            if 0 <= nx < rows and 0 <= ny < cols:
                if maze[nx][ny] == 0:       # nếu không phải tường
                    
                    neighbor = (nx, ny)
                    new_g = g[current] + 1  # mỗi bước tốn 1
                    
                    # nếu chưa thấy hoặc tìm được đường tốt hơn
                    if neighbor not in g or new_g < g[neighbor]:
                        g[neighbor] = new_g
                        
                        # f = g + h
                        f = new_g + heuristic(neighbor, goal)
                        heapq.heappush(OPEN, (f, neighbor))
                        
                        parent[neighbor] = current
    
    return parent


In [4]:
def reconstruct_path(parent, start, goal):
    """
    Trả về danh sách đỉnh từ start → goal
    """
    if goal not in parent:
        return []
    path = []
    current = goal
    
    while current is not None:
        path.append(current)
        current = parent[current]
    
    return path[::-1]   # đảo ngược lại

In [5]:
maze = [
    [0, 0, 0, 1, 0, 0],
    [1, 1, 0, 1, 0, 1],
    [0, 0, 0, 0, 0, 1],
    [0, 1, 1, 1, 0, 0],
    [0, 0, 0, 1, 1, 0],
    [1, 1, 0, 0, 0, 0]
]
start = (0, 0)
goal  = (5, 5)
parent = Astar_maze(maze, start, goal)
path = reconstruct_path(parent, start, goal)

print("Đường đi tìm được:")
print(path)

Đường đi tìm được:
[(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (2, 4), (3, 4), (3, 5), (4, 5), (5, 5)]


## Bài toán 8-puzzle

In [6]:
def get_neighbors(state):
    """
    state: tuple 9 phần tử
    Trả về danh sách trạng thái kề
    """
    neighbors = []
    
    idx = state.index(0)           # vị trí ô trống
    row, col = divmod(idx, 3)
    
    # 4 hướng di chuyển
    moves = [(-1,0),(1,0),(0,-1),(0,1)]
    
    for dx, dy in moves:
        nr, nc = row + dx, col + dy
        
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_idx = nr*3 + nc
            
            # hoán đổi 0 với ô bên cạnh
            new_state = list(state)
            new_state[idx], new_state[new_idx] = new_state[new_idx], new_state[idx]
            
            neighbors.append(tuple(new_state))
    
    return neighbors


In [7]:
def heuristic_puzzle(state, goal):
    """
    Tổng khoảng cách Manhattan của từng ô (1→8)
    """
    distance = 0
    
    for num in range(1, 9):
        i = state.index(num)
        gi = goal.index(num)
        
        x1, y1 = divmod(i, 3)
        x2, y2 = divmod(gi, 3)
        
        distance += abs(x1 - x2) + abs(y1 - y2)
    
    return distance

In [8]:
def Astar_puzzle(start, goal):
    """
    start, goal: tuple 9 phần tử
    """
    
    OPEN = []
    heapq.heappush(OPEN, (0, start))
    
    g = {start: 0}                 # chi phí thật
    parent = {start: None}         # truy vết
    CLOSED = set()
    
    while OPEN:
        _, current = heapq.heappop(OPEN)
        
        if current in CLOSED:
            continue
        
        if current == goal:
            break
        
        CLOSED.add(current)
        
        # duyệt các trạng thái kề
        for neighbor in get_neighbors(current):
            
            new_g = g[current] + 1   # mỗi bước tốn 1
            
            # nếu tìm được đường tốt hơn
            if neighbor not in g or new_g < g[neighbor]:
                g[neighbor] = new_g
                
                f = new_g + heuristic_puzzle(neighbor, goal)
                heapq.heappush(OPEN, (f, neighbor))
                
                parent[neighbor] = current
    
    return parent

In [23]:
goal = (1,2,3,
        4,5,6,
        7,8,0)
start = (1,2,3,
         4,5,6,
         7,0,8)
parent = Astar_puzzle(start, goal)

path = []
current = goal

while current is not None:
    path.append(current)
    current = parent.get(current)

path.reverse()

print("Số bước:", len(path)-1)
print("Đường đi:")

for state in path:
    print(state)


Số bước: 1
Đường đi:
(1, 2, 3, 4, 5, 6, 7, 0, 8)
(1, 2, 3, 4, 5, 6, 7, 8, 0)
